## PhysProp FP analysis notebook

For GEN-1106

In [ ]:
# setup
import sys
if "../../.." not in sys.path:
    sys.path.append("../../..")
import requests
from IPython.display import JSON
from genraweb.resources import DB
from genraweb.lib.fp.fpclass import FPGen
from rdkit import Chem
from rdkit.Chem import Descriptors
from genraweb.lib.mongofp_NN import searchFP, searchColByFP, DS, COL
from genraweb.lib.fp.nn_lookup import chem_nn, fp_n_for_chems
from deepdiff import DeepDiff
from genraweb.lib import multitarget
from genraweb.lib.chem_id import NOTNAMES, UNNAMED, ChemID
from genraweb.lib.fp.fputils import FP_INFO, parse_fp, physchem_slow
from genraweb.resources import TOXREF_SIZE, redis_cache

API_URL = "http://genra_api:5000/genra-api/api/genra/"

## Health check

No Python API

In [ ]:
JSON(requests.get(API_URL + "v3/healthCheck/").json())

## FP IDs

Depends on deployment type

In [ ]:
FPGen.FPClass

In [ ]:
# JSON(next(DB.compounds.find({"name": "Bisphenol A"}, {"_id": 0})))

In [ ]:
fp = FPGen.FPClass["chm_phch"](DB, FPGen.FPClass["chm_phch"].fp_output_basename)
fp.bit_names()
possible = set(fp.all_chem_ids())
len(possible)
actual = set(i["dsstox_cid"] for i in fp.DB[fp.fp_output_basename].find({"dsstox_cid": {"$ne": None}}))

In [ ]:
print(len(possible)-len(actual))
oneid = (possible-actual).pop()  
print(oneid)
one = DB.compounds.find_one({"dsstox_cid": oneid})
print(one)
# DB.compounds.find_one({"dsstox_cid": "DTXCID30182"})
mol = Chem.MolFromSmiles(one["smiles"])
print(Descriptors.NumHDonors(mol))
print(Descriptors.NumHAcceptors(mol))
print(Chem.Descriptors.ExactMolWt(mol))
print(bool(mol))
print("FP", DB.physchem_fp.find_one({"dsstox_cid": oneid}))
print(DB.physprop.find_one({"dsstox_cid": oneid}))


In [ ]:
logp = set(
 i["dsstox_cid"]
 for i in DB["physprop"].find(
     {
         "dsstox_cid": {"$ne": None},
         "predicted_props.OPERA_LogP": {"$ne": None},
     },
     {"_id": False, "dsstox_cid": True},
 )
)
hbad = set(
  i["dsstox_cid"]
 for i in DB["compounds"].find(
      {
         "dsstox_cid": {"$ne": None},
         "HBA": {"$ne": None},
         "HBD": {"$ne": None},
     },
     {"_id": False, "dsstox_cid": True},
 )
)
cid = set(
  i["dsstox_cid"]
 for i in DB["compounds"].find(
      {
         "dsstox_cid": {"$ne": None},
     },
     {"_id": False, "dsstox_cid": True},
 )
)


In [ ]:
print(len(logp-hbad))
print(len(logp-cid))
print(len(logp-hbad) - len(logp-cid))
one = (logp-hbad).pop()
print(one)
one = DB.compounds.find_one({"dsstox_cid": one})
print(one)
# DB.compounds.find_one({"dsstox_cid": "DTXCID30182"})
mol = Chem.MolFromSmiles(one["smiles"])
print(Descriptors.NumHDonors(mol))
print(Descriptors.NumHAcceptors(mol))
print(Chem.Descriptors.ExactMolWt(mol))
print(bool(mol))

In [ ]:
possible = fp.all_chem_ids()
existing = set(
  i["dsstox_cid"]
 for i in DB["physchem_fp"].find(
      {
         "dsstox_cid": {"$ne": None},
     },
     {"_id": False, "dsstox_cid": True},
 )
)
possible - existing

In [ ]:
fp = "chm_mrgn"
a = searchColByFP("DTXCID30182", 0.1, 8, DB, COL.get(fp), DS.get(fp), FPGen.FPClass[fp], simple=True, sel_by="tox_txrf")
fp = "chm_phch"
b = searchColByFP("DTXCID30182", 0.1, 9, DB, COL.get(fp), DS.get(fp), FPGen.FPClass[fp], simple=False, sel_by="tox_txrf")
# del a[0]
print(" ".join(i["dsstox_cid"] for i in a))
print(" ".join(i["dsstox_cid"] for i in b))
print(" ".join(str(i["jaccard"]) for i in a))
print(" ".join(str(i["jaccard"]) for i in b))
JSON({"ab":{"a":a, "b":b}}) # ,"diff":DeepDiff(a, b)})

In [ ]:
a = searchFP("DTXCID30182", "chm_phch_W1_and_chm_mrgn_W1", DB=DB, max_hits=10, simple=True)
b = searchFP("DTXCID30182", "chm_httr_W1_and_chm_mrgn_W1", DB=DB, max_hits=10, simple=False)
print(" ".join(i["dsstox_cid"] for i in a))
print(" ".join(i["dsstox_cid"] for i in b))
print(" ".join(str(i["jaccard"]) for i in a))
print(" ".join(str(i["jaccard"]) for i in b))
# JSON({"ab":{"a":a, "b":b},"diff":DeepDiff(a, b)})
JSON({"ab":{"a":a, "b":b}})

In [ ]:
JSON(chem_nn("DTXCID30182", "chm_mrgn", "tox_txrf", 0.1, 100))

In [ ]:
chem_id_in = "DTXCID30182"
fp = "chm_phch_W1_and_chm_mrgn_W1"
DB=DB
max_hits=10
simple=False
s0=0.1
sel_by = "tox_txrf"
kwargs = {}

chem_id = multitarget.clean_id(chem_id_in)
assert not multitarget.is_multi(chem_id)
target_chem_id, chem = ChemID.promote_id(chem_id_in)
fps, weights = parse_fp(fp)

lower_bound = 100 if simple else TOXREF_SIZE
max_hits_search = max(kwargs.get("max_hits", 0), lower_bound)

print(fps, max_hits_search)

In [ ]:
neighborhoods_for_fps = []
for fp in fps:
    col = COL.get(fp)
    ds = DS.get(fp)
    if not (ds and col):
        pass
        # return []

    neighborhood_for_fp = searchColByFP(
        target_chem_id=target_chem_id,
        s0=s0,
        max_hits_search=max_hits_search,
        DB=DB,
        col=col,
        fpn=ds,
        fp=FPGen.FPClass[fp],
        sel_by=sel_by,
        simple=simple,
    )

    neighborhoods_for_fps.append(neighborhood_for_fp)

In [ ]:
fp = 'chm_phch'
col = COL.get(fp)
ds = DS.get(fp)
neighborhood_for_fp = searchColByFP(
    target_chem_id=target_chem_id,
    s0=s0,
    max_hits_search=101,
    DB=DB,
    col=col,
    fpn=ds,
    fp=FPGen.FPClass[fp],
    # sel_by=sel_by,
    sel_by="no_filter",
    simple=simple,
)
JSON(neighborhood_for_fp)

In [ ]:
filter_collection = FPGen.FPClass[sel_by].output_collection_name()
filter_collection = FPGen.FPClass[sel_by].fp_output_basename
filter_collection
projection = ChemID.chem_id_proj(include_core_fields=True)
in_filter = set(
  i["chem_id"] for i in DB[filter_collection].find({}, projection)
)
len(in_filter)


# OTHER CODE BELOW HERE

But better to use `search_chems` which searches synonyms etc. etc.

In [ ]:
from genraweb.routes.searchChem_grouped import search_chems
JSON(search_chems("BPA"))  # RDKit complains about BPA not being SMILES, ignore it

...and the corresponding REST call...

In [ ]:
JSON(requests.get(API_URL + "v3/searchChems/?txt=BPA").json())

## Setup

No Python API

In [ ]:
JSON(requests.get(API_URL + "v4/uiSetup?chem_id=DTXCID30182").json())

## Radial view

Note results are not quite the same

In [ ]:
from genraweb.lib.mongofp_NN import searchFP
nghbrs = searchFP("DTXCID30182", fp="chm_mrgn", sel_by="tox_txrf", s0=0.1, max_hits=10 + 1)
JSON(nghbrs)

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiRadialView?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## FP heat map

Now Python and REST API results very different, REST result AG Grid flavored

In [ ]:
from genraweb.lib.fp.fputils import fp_counts_for_chems
chem_ids = [i["chem_id"] for i in nghbrs]
JSON(fp_counts_for_chems(chem_ids=chem_ids))

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiFingerPrintHeatChart?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## Assay list (panel 3)

In [ ]:
from genraweb.lib.fp.fputils import get_toxref_assays_for_chems, readacross_table
# readacross_table is ~ the uiAssayList REST result
# readacross_table(target_chem_id="DTXCID30182", fp_id="chm_mrgn", sel_by="tox_txrf", s0=0.1, k0=10 + 1))
get_toxref_assays_for_chems(chem_ids=chem_ids)  # Pandas DF

In [ ]:
JSON(requests.get(
    API_URL + "v4/uiAssayList?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json())

## Generate Read Across

Python API wise this would just be `get_toxref_assays_for_chems()` again as above, apart from PhysProp data

In [ ]:
from genraweb.lib.properties.physprop import ID2PP, chem_props
ID2PP

In [ ]:
JSON(chem_props(chem_ids))

In [ ]:
gra_data = requests.get(
    API_URL + "v4/uiGenerateReadAcross?chem_id=DTXCID30182&fp=chm_mrgn&sel_by=tox_txrf&s0=0.1&k0=10"
).json()
JSON(gra_data)

## Run read-across

In [ ]:
from genraweb.lib.genrapred import runGenRA
predictions = runGenRA(
    "DTXCID30182",
    CID=chem_ids,
    DB=DB,
    fp_x="chm_mrgn",
    fp_y="toxp_txrf",
    sel_by="tox_txrf",
    metric="jaccard",
    k0=10,
    s0=0.1,
    pred=True,
    ret="df",
    n_perm=200,
    pos_min=1,
    neg_min=1,
)
JSON(predictions)

In [ ]:
post_data = {
    "fp": "chm_mrgn",
    "k0": 10,
    "s0": 0.1,
    "dsstox_cid": "DTXCID30182",
    "sel_by": "tox_txrf",
    "neg0": 1,
    "pos0": 1,
    "chem_inc": [
        {"isChecked": True, "chem_id": i} for i in chem_ids
    ],
    "tox_inc": [],  # which assays,  [] => all
}
rra_data = requests.post(
    API_URL + "v4/uiRunReadAcross", json=post_data
).json()
JSON(rra_data)

## Download

In [ ]:
post_data = {
    "fp": "chm_mrgn",
    "k0": 10,
    "s0": 0.1,
    "chem_id": "DTXCID30182",
    "sel_by": "tox_txrf",
    "neg0": 1,
    "pos0": 1,
    "chem_inc": [
        {"isChecked": True, "chem_id": i} for i in chem_ids
    ],
    "tox_inc": [],  # which assays,  [] => all
    "rra": True,  # False for "Generate Read Across version"
}
download = requests.post(
    API_URL + "v4/uiDownload/xlsx", json=post_data
)
from io import BytesIO
dl = BytesIO(download.content)
dl.seek(0)
import pandas as pd
pd.read_excel(dl, sheet_name="Metadata")